# Notebook 2 — Community Detection and Removing Harry
**YTS+ DSEB 2026 · Project A · Harry Potter**

---

## The question for this session

The algorithm knows nothing about Harry Potter. It sees only edge weights — which characters appear together and how often. It will partition the network into groups.

Does it recover the factions Rowling built?

And then: **remove Harry entirely.** Who holds each world together without him?

---
## Setup

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import community as community_louvain
from collections import defaultdict
import time
import os

os.makedirs('images', exist_ok=True)

DATA = 'data/'
print('Ready.')

---
## Part 1 — EDA: What's in the pre-computed community data?

We pre-ran community detection for Books 3, 5, and 7 — with Harry and without Harry — and saved the results. Let's look at what we have before running anything ourselves.

In [ ]:
# Load the pre-computed community analysis
ca = pd.read_csv(DATA + 'community_analysis.csv')

print(f'Shape: {ca.shape}')
print(f'Columns: {ca.columns.tolist()}')
print()
ca.head(8)

In [ ]:
# How many characters per book? How many communities?
for book in [3, 5, 7]:
    b = ca[ca['book'] == book]
    n_chars = len(b)
    n_comm_with = b['community_with_harry'].nunique()
    n_comm_without = b['community_without_harry'].nunique()
    print(f'Book {book}: {n_chars} characters | {n_comm_with} communities (with Harry) | {n_comm_without} communities (without Harry)')

In [ ]:
# Community sizes — how big are the communities in Book 5 (with Harry)?
b5 = ca[ca['book'] == 5]
sizes = b5.groupby('community_with_harry').size().sort_values(ascending=False)

print('Book 5 community sizes (with Harry):')
print(sizes.to_string())
print(f'\nLargest community: {sizes.iloc[0]} characters')
print(f'Smallest: {sizes.iloc[-1]} characters')

In [ ]:
# Where is Harry in each book?
harry_rows = ca[ca['character'] == 'Harry Potter'][['book', 'community_with_harry', 'community_size_with_harry',
                                                      'community_without_harry', 'community_size_without_harry']]
print('Harry Potter across books:')
print(harry_rows.to_string(index=False))
print()
print('Note: community_without_harry is the community he would be assigned to if he were present')
print('In Session 3 we actually remove him and recompute.')

---
## Part 2 — Before running: predict the communities

Draw or write the communities you expect Louvain to find in Book 5. Name each community. Who leads each one?

The algorithm sees only edge weights. No names. No plot. It finds groups of characters who appear together more than they appear with outsiders.

**My expected communities in Book 5:**

| Community name | Expected members (a few) | Expected leader |
|---|---|---|
| | | |
| | | |
| | | |
| | | |

---
## Part 3 — Run Louvain on Book 3

In [ ]:
# Load and build Book 3
df3 = pd.read_csv(DATA + 'hp_book3_edges.csv')
G3 = nx.from_pandas_edgelist(df3, 'source', 'target', edge_attr='weight')
print(f'Book 3: {G3.number_of_nodes()} characters, {G3.number_of_edges()} pairs')

In [ ]:
# Run community detection — this is fast (< 1 second)
t0 = time.time()
partition3 = community_louvain.best_partition(G3, weight='weight', random_state=42)
print(f'Done in {time.time()-t0:.3f}s')

n_communities = max(partition3.values()) + 1
print(f'Communities found: {n_communities}')

In [ ]:
# Show top characters in each community, sorted by within-community degree
degree3 = dict(G3.degree(weight='weight'))
comm_members3 = defaultdict(list)
for char, cid in partition3.items():
    comm_members3[cid].append(char)

# Sort communities by size, show largest first
for cid in sorted(comm_members3, key=lambda c: -len(comm_members3[c])):
    members = comm_members3[cid]
    if len(members) < 3:
        continue
    top = sorted(members, key=lambda n: degree3.get(n, 0), reverse=True)[:6]
    print(f'Community {cid} ({len(members)} characters):')
    print(f'  {" | ".join(top)}')
    print()

**For each community with 4+ members, name it.** What is this group's identity in the story?

**Key thing to find:** Look for the community containing Sirius, Lupin, Snape, and Pettigrew. The algorithm found the secret history of Book 3 from scene construction alone — it never read the books.

| Community ID | Top members | What HP world is this? |
|---|---|---|
| | | |
| | | |
| | | |

**Write:** Which community surprised you most, and why?

**My answer:**

*Write here.*

---
## Part 4 — Louvain on Books 5 and 7

In [ ]:
# Book 5
df5 = pd.read_csv(DATA + 'hp_book5_edges.csv')
G5 = nx.from_pandas_edgelist(df5, 'source', 'target', edge_attr='weight')

t0 = time.time()
partition5 = community_louvain.best_partition(G5, weight='weight', random_state=42)
print(f'Book 5 — {G5.number_of_nodes()} characters | {max(partition5.values())+1} communities | {time.time()-t0:.3f}s')

degree5 = dict(G5.degree(weight='weight'))
comm_members5 = defaultdict(list)
for char, cid in partition5.items():
    comm_members5[cid].append(char)

print()
for cid in sorted(comm_members5, key=lambda c: -len(comm_members5[c])):
    members = comm_members5[cid]
    if len(members) < 4:
        continue
    top = sorted(members, key=lambda n: degree5.get(n, 0), reverse=True)[:6]
    print(f'Community {cid} ({len(members)} chars): {" | ".join(top)}')

harry_comm5 = partition5.get('Harry Potter')
print(f'\nHarry is in Community {harry_comm5} (size {len(comm_members5[harry_comm5])})')

In [ ]:
# Book 7
df7 = pd.read_csv(DATA + 'hp_book7_edges.csv')
G7 = nx.from_pandas_edgelist(df7, 'source', 'target', edge_attr='weight')

t0 = time.time()
partition7 = community_louvain.best_partition(G7, weight='weight', random_state=42)
print(f'Book 7 — {G7.number_of_nodes()} characters | {max(partition7.values())+1} communities | {time.time()-t0:.3f}s')

degree7 = dict(G7.degree(weight='weight'))
comm_members7 = defaultdict(list)
for char, cid in partition7.items():
    comm_members7[cid].append(char)

print()
for cid in sorted(comm_members7, key=lambda c: -len(comm_members7[c])):
    members = comm_members7[cid]
    if len(members) < 4:
        continue
    top = sorted(members, key=lambda n: degree7.get(n, 0), reverse=True)[:6]
    print(f'Community {cid} ({len(members)} chars): {" | ".join(top)}')

harry_comm7 = partition7.get('Harry Potter')
print(f'\nHarry is in Community {harry_comm7} (size {len(comm_members7[harry_comm7])})')

**Things to find in Book 7:**
- Where are Snape and Dumbledore? Are they in the Horcrux hunt community?
- Which community is Luna in? Who else is there?

**Fill in the table:**

| Book | Harry's community | Who is with him? | Who is NOT with him? |
|---|---|---|---|
| Book 3 | | | |
| Book 5 | | | |
| Book 7 | | | |

**Write:** Harry's community changes every book. What does this tell you about Rowling's narrative structure? Which book pulls Harry furthest from his friends?

**Your answer:**

*Write here.*

---
## Part 5 — Remove Harry

Harry is in almost every scene. His presence dominates the network. When we remove him, the other characters' roles become visible on their own terms.

**Before loading:** Predict — without Harry, who leads each world in Book 5?
- The Order of the Phoenix world:
- The Ron/Hermione/school world:
- The Slytherin world:

**My predictions (no Harry, Book 5):**

- Order world leader: ___
- School world leader: ___
- Slytherin world leader: ___

In [ ]:
# The pre-computed file has within-community degree with and without Harry
ca = pd.read_csv(DATA + 'community_analysis.csv')

# Key columns we care about:
# community_without_harry — which community this character falls into when Harry is removed
# within_degree_without_harry — how many co-appearances they have within that community

print(ca[['book', 'character', 'community_without_harry', 'within_degree_without_harry']].head(10).to_string(index=False))

### Finding 1: The Slytherin world — who really holds it together?

Global degree in Book 3: Draco is #8 overall. Goyle is #22. Crabbe is #21.

Within the Slytherin clique, with Harry removed — what happens to those rankings?

In [ ]:
b3 = ca[ca['book'] == 3].copy()

# Find Draco's community (without Harry)
draco_comm = b3[b3['character'] == 'Draco Malfoy']['community_without_harry'].values[0]

# All members of that community, ranked within it
slytherin = b3[b3['community_without_harry'] == draco_comm].copy()
slytherin = slytherin.sort_values('within_degree_without_harry', ascending=False).reset_index(drop=True)
slytherin['within_rank'] = slytherin.index + 1

print(f'Slytherin community (Book 3, no Harry) — {len(slytherin)} characters')
print()
print(slytherin[['within_rank', 'character', 'global_degree', 'within_degree_without_harry']].head(8).to_string(index=False))

**What did you find?**

Rowling named Draco the leader of his clique. The global degree ranking agrees — Draco is #8 overall, well above Goyle (#22) and Crabbe (#21).

Within the Slytherin community with Harry removed — who is #1? Who is #2? Where is Draco?

**You must take a position:**
- Either the algorithm missed something — explain what.
- Or it is right — explain what Rowling actually did in her scene construction vs. how she presented Draco.

**Within-community ranking (Book 3, no Harry):**

1. 
2. 
3. 

**My interpretation:**

*Write here.*

### Finding 2: The Order world — Sirius or Dumbledore?

In [ ]:
b5 = ca[ca['book'] == 5].copy()

# Sirius's community without Harry
sirius_comm = b5[b5['character'] == 'Sirius Black']['community_without_harry'].values[0]

order_world = b5[b5['community_without_harry'] == sirius_comm].sort_values(
    'within_degree_without_harry', ascending=False).reset_index(drop=True)
order_world['within_rank'] = order_world.index + 1

print(f'Order world (Book 5, no Harry) — {len(order_world)} characters')
print()
print(order_world[['within_rank', 'character', 'within_degree_without_harry']].head(10).to_string(index=False))

# Where is Dumbledore?
print()
dumbledore_comm = b5[b5['character'] == 'Albus Dumbledore']['community_without_harry'].values[0]
print(f'Dumbledore is in community {dumbledore_comm}')
print(f'Sirius is in community {sirius_comm}')
print(f'Same community: {sirius_comm == dumbledore_comm}')

**Sirius vs Dumbledore:**

Without Harry, who holds the Order together? And where does Dumbledore end up — deeply embedded in one world, or spread across several?

**Write:** What does this tell you about how Rowling wrote Dumbledore versus how she wrote Sirius? One is a connector who spans many worlds. The other is a pillar who holds one world together.

**My interpretation:**

*Write here.*

### Finding 3: Hermione vs Ron — across every book

In [ ]:
print('Hermione vs Ron — within-community rank (without Harry)\n')

for book in [3, 5, 7]:
    b = ca[ca['book'] == book].copy()
    herm_comm = b[b['character'] == 'Hermione Granger']['community_without_harry'].values[0]
    
    community = b[b['community_without_harry'] == herm_comm].sort_values(
        'within_degree_without_harry', ascending=False).reset_index(drop=True)
    community['rank'] = community.index + 1
    
    for name in ['Hermione Granger', 'Ronald Weasley']:
        row = community[community['character'] == name]
        if len(row) > 0:
            r = int(row['rank'].values[0])
            wd = int(row['within_degree_without_harry'].values[0])
            print(f'Book {book} | {name:<22} within-rank #{r:<3} within-degree {wd}')
    print()

In [ ]:
# Plot Hermione vs Ron across books
books = [3, 5, 7]
herm_ranks, ron_ranks = [], []

for book in books:
    b = ca[ca['book'] == book].copy()
    herm_comm = b[b['character'] == 'Hermione Granger']['community_without_harry'].values[0]
    community = b[b['community_without_harry'] == herm_comm].sort_values(
        'within_degree_without_harry', ascending=False).reset_index(drop=True)
    community['rank'] = community.index + 1
    herm_ranks.append(community[community['character'] == 'Hermione Granger']['rank'].values[0])
    ron_ranks.append(community[community['character'] == 'Ronald Weasley']['rank'].values[0])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1, 2], herm_ranks, 'o-', color='purple', linewidth=2, markersize=10, label='Hermione')
ax.plot([0, 1, 2], ron_ranks, 's--', color='darkorange', linewidth=2, markersize=10, label='Ron')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Book 3', 'Book 5', 'Book 7'], fontsize=12)
ax.invert_yaxis()
ax.set_ylabel('Within-community rank (1 = highest)', fontsize=11)
ax.set_title('Hermione vs Ron — within-trio rank (no Harry)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('images/hermione_ron_rank.png', dpi=150)
plt.show()

**The most arguable finding in this project.**

Hermione ranks above Ron within the trio community in every single book — Books 3, 5, and 7.

You must take a position and defend it with evidence from the books:
- **Option A:** The data is right. Rowling actually wrote Hermione as the structural center of the trio, regardless of how the narrative presents Ron.
- **Option B:** The data is wrong. Explain what the algorithm cannot see that would change the result.

Write 4–5 sentences. Reference specific scenes or plot moments.

**My position on Hermione > Ron:**

*Write here — take a position and defend it.*

---
## What to bring to the next session

- Your filled-in tables and written answers from this notebook
- The finding that surprised you most
- Your Hermione > Ron argument — you will need to defend it

**Next:** Notebook 3 — Predictions and Refusals. The network will predict which characters Rowling should have put in scenes together. Some of those meetings never happened. We will ask: which refusal would you write into the movies?